# PixiJoy Depth (M2) — Kaggle GPU (quality)

**Settings (right panel):** Accelerator → **GPU T4** (or P100) → Internet **ON** → Save.

Kaggle free tier ≈ **30 GPU hours / week**. Sessions also have a **max duration** —
exit **137** means the host killed the notebook (timeout/OOM), not a Python assert.

**Chunked train:** each Run All does `SESSION_EPOCHS` (default **8**), then exports.
Re-run with resume until early-stop or full budget.

**Before the session dies:** *Save Version* (Save & Run All) **or** download
`dist/*.ckpt.keras` + `*.last.keras` + `train_state.json`. Next notebook → Add Input → that output.


In [ ]:
!nvidia-smi || true
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("TF", tf.__version__, "GPUs", gpus, flush=True)
assert gpus, 'Enable GPU: Notebook settings → Accelerator → GPU T4 → Save → Restart'


In [ ]:
import os, sys, urllib.request, zipfile, shutil
from pathlib import Path

ROOT = Path('/kaggle/working/pixijoy_depth')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
(ROOT / 'dist').mkdir(exist_ok=True)
(ROOT / 'data').mkdir(exist_ok=True)
print('cwd', ROOT, flush=True)

KERAS_NAME = 'depth.keras'
CKPT_NAME = 'depth.ckpt.keras'
LAST_NAME = 'depth.last.keras'

# Optional: copy prior ckpt from a Kaggle Dataset Input
for inp in Path('/kaggle/input').glob('*'):
    for name in (KERAS_NAME, CKPT_NAME, LAST_NAME, 'train_state.json'):
        src = next(inp.rglob(name), None)
        if src and src.is_file():
            dst = ROOT / 'dist' / src.name
            shutil.copy2(src, dst)
            print('restored from input', src, '->', dst, flush=True)

# Optional: HTTP resume pack (no Add Input needed)
RESUME_URL = os.environ.get('RESUME_URL', '').strip()
if not RESUME_URL and 'depth' == 'depth':
    RESUME_URL = 'https://github.com/Saikumar-S0906/Release-/releases/download/ml-packs-v1/pixijoy_depth_resume.zip'
if RESUME_URL and not (ROOT / 'dist' / CKPT_NAME).is_file():
    print('fetching resume', RESUME_URL, flush=True)
    rz = ROOT / 'resume.zip'
    urllib.request.urlretrieve(RESUME_URL, rz)
    with zipfile.ZipFile(rz) as z:
        z.extractall(ROOT / 'dist')
    print('resume extracted', list((ROOT / 'dist').iterdir()), flush=True)

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/depth_lab.zip'
LAB_ZIP_VERSION = '20260929d'
urllib.request.urlretrieve(ZIP_URL + f'?v={LAB_ZIP_VERSION}', 'depth_lab.zip')
with zipfile.ZipFile('depth_lab.zip') as z:
    z.extractall('.')
assert Path('train.py').is_file()
print('lab zip OK', flush=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path

def run_stream(cmd):
    print('STREAM', ' '.join(map(str, cmd)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    print(f'[exit {rc}]', flush=True)
    return rc

rc = run_stream([sys.executable, '-u', 'prepare_quality_data.py', '--data', './data'])
assert rc == 0
assert (Path('data') / 'READY').is_file()
print('data READY ok', flush=True)


In [ ]:
import os, hashlib, subprocess, sys
from pathlib import Path

# Keep each interactive session under Kaggle max duration (exit 137 = killed).
SESSION_EPOCHS = 8

OUT = Path('dist') / 'depth.keras'
ckpt = OUT.with_suffix('.ckpt.keras')
last = OUT.with_name(OUT.stem + '.last.keras')

def run_stream(cmd):
    print('STREAM', ' '.join(map(str, cmd)), flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

cmd = [
    sys.executable, '-u', 'train.py', '--quality', '--out', str(OUT),
    '--session-epochs', str(SESSION_EPOCHS),
]
if ckpt.is_file() or last.is_file():
    cmd.append('--resume')
    print('will resume', ckpt if ckpt.is_file() else last, flush=True)
rc = run_stream(cmd)
assert rc == 0
rc = run_stream([sys.executable, '-u', 'export_tflite.py', '--keras', str(OUT), '--out', 'dist/depth.tflite'])
assert rc == 0
rc = run_stream([sys.executable, '-u', 'verify_contract.py', '--quality', '--soft-mae', '--data', './data', '--model', 'dist/depth.tflite'])
assert rc == 0, 'contract (shapes/dtype) failed'
p = Path('dist/depth.tflite')
print('SHIP_OK', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest(), flush=True)
print('Save Version now (or download dist/) before the session times out.', flush=True)
print('If you saw SESSION_CHUNK_DONE, re-run this notebook with the Output as Input.', flush=True)
